# Realistic clouds from observations (solar spectral)

Uses the cloud layers of {doc}`radiosonde_thermal` (lidar and radar, HALO-(AC)³ RF13, 2022-04-04) for spectral solar simulations between 1024 and 2400 nm, the range of the specMACS SWIR camera. It shows how the observed clouds change the upward and downward spectral irradiance at flight level. It is not a validation: comparing with specMACS needs nadir radiances (`uu`) at the specMACS viewing geometry and a radiometric comparison that is not done here.

```{note}
This notebook needs HALO-(AC)³ campaign data from `/projekt_agmwend` (VELOX, lidar, radar) and the dropsondes in `radiosonde/`, none of which are public. It is not executed for the book and has no stored outputs; run it with the campaign data, or replace the file paths with your own observations.
```

In [ ]:
import pyradtran
from pyradtran import load_config
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pathlib import Path
import logging

logging.getLogger('pyradtran').setLevel(logging.CRITICAL)

cfg = load_config()
cfg.simulation_defaults.rte_solver = "disort"
cfg.simulation_defaults.mol_abs_param = "reptran coarse"
cfg.simulation_defaults.source = "solar"
cfg.simulation_defaults.wavelength_nm = [1024, 2400]      # specMACS SWIR range
cfg.simulation_defaults.integrate_wavelength = False
cfg.simulation_defaults.albedo_value = 0.3                # assumption; snow albedo drops strongly beyond 1.4 µm
cfg.simulation_defaults.output_columns = ["zout", "lambda", "sza", "edir", "edn", "eup"]
cfg.simulation_defaults.h2o_source = "radiosonde"
cfg.simulation_defaults.ozone_du = 300.0
cfg.paths.radiosonde_base = "radiosonde"                  # dropsondes, not shipped
cfg.execution.max_workers = 8
cfg.execution.timeout_seconds = 120
cfg.output.filename_prefix = "radiosonde_solar_spectral"
config_path = Path("config/radiosonde_solar_spectral.yaml")
cfg.to_yaml(config_path)

## Load observation data

The VELOX channel 3 file (10.74 µm, RF13) provides the flight track and a thermal view of the clouds; `BT_2D` and `BT_Center` are in °C. specMACS is shown as a quick look only.

In [ ]:
ds = xr.open_dataset(
    "/projekt_agmwend/data/HALO-AC3/02_Flights/HALO-AC3_20220404_HALO_RF13/VELOX/VELOX_327kveL/Processed/to_publish/final/HALO-AC3_HALO_VELOX_BT3_10740nm_20220404_RF13_v3.0.nc"
).sel(time=slice('2022-04-04T13:41:00', '2022-04-04T14:00:00')).load()

ds.BT_Center.plot(figsize=(20, 3))

In [ ]:
ds_specMACS = xr.open_zarr(
    "/projekt_agmwend/data/HALO-AC3/02_Flights/HALO-AC3_20220404_HALO_RF13/specMACS/specMACS_RF13.zarr/"
).sel(time=slice('2022-04-04T13:41:00', '2022-04-04T14:00:00')).load()

# Quick look at the specMACS radiance at its first wavelength
ds_specMACS.radiance.isel(wavelength=0).plot(x='time', figsize=(20, 3))

In [ ]:
# One VELOX frame and its brightness-temperature histogram
time_idx = 240

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
bins = np.linspace(-25, 0, 100)
ds.BT_2D.isel(time=time_idx).plot(cmap='viridis', vmin=-25, vmax=0, ax=ax[1],
                                  cbar_kwargs={'label': 'Brightness temperature (°C)'})
ds.BT_2D.isel(time=time_idx).plot.hist(ax=ax[0], bins=bins, color='gray', alpha=0.7, histtype='step', lw=2)
ax[0].set_xlabel('Brightness temperature (°C)')
ax[0].set_ylabel('Count')
plt.tight_layout()
plt.show()

## Prepare the flight track

Only the track goes into the simulation: every dimension of the input dataset is a loop, so the image dimensions `x`, `y` must not be passed. The aircraft altitude is a data variable, so each time step is simulated at flight level ({doc}`/guide/concepts`).

In [ ]:
track = xr.Dataset({
    'latitude': ds['lat'],
    'longitude': ds['lon'],
    'altitude': ds['alt'] / 1000,   # m -> km, flight level per time step
})
track

## Clear-sky simulation

Spectral irradiance at flight level without clouds.

In [ ]:
ds_sim_clear = track.pyradtran.run(
    config_path=config_path,
    save_to_file=False,
    show_progress=False,
)
ds_sim_clear

## Cloud layers from lidar and radar

Cloud top and base come from the lidar backscatter ratio (> 100 counts as cloud); the radar reflectivity is shown for context. Each cloudy time step gets one homogeneous liquid layer with 0.1 g m⁻³ and the default effective radius of 10 µm ({doc}`/guide/clouds`). The water content is an assumption, not a retrieval.

In [ ]:
lidar = xr.open_dataset("/projekt_agmwend/home_rad/Joshua/HALO-AC3_unified_data/unified_bsrgl.nc").sel(time=ds.time).load()
lidar = lidar.where(lidar.backscatter_ratio > 100)
radar = xr.open_dataset("/projekt_agmwend/home_rad/Joshua/HALO-AC3_unified_data/unified_radar.nc").sel(time=ds.time).load()

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw={'width_ratios': [1, .2]})
lidar.backscatter_ratio.plot(y='altitude', robust=True, add_colorbar=False, ax=ax)
radar.dBZg.plot(y='height', robust=True, cmap='gray_r', alpha=.7, add_colorbar=False, ax=ax)
ax.set_ylim(0, 2000)


def cth_and_layer_thickness(ds, ds_lidar=lidar):
    """Cloud top height, thickness (m) and cloud fraction from lidar data
    that are already masked to backscatter_ratio > 100."""
    ds_lidar_sel = ds_lidar.sel(time=ds.time, method='nearest')
    cloudy = ds_lidar_sel.backscatter_ratio.notnull()
    cth = ds_lidar_sel.altitude.where(cloudy).max(dim='altitude')
    cbh = ds_lidar_sel.altitude.where(cloudy).min(dim='altitude')
    # Ignore returns below 150 m (surface)
    cth = cth.where(cth > 150)
    cbh = cbh.where(cbh > 150)
    cloudmask = (cloudy & (ds_lidar_sel.altitude > 150)).mean(dim='altitude')
    return xr.Dataset({'cth': cth, 'thickness': cth - cbh, 'cloudmask': cloudmask})


cth_ds = cth_and_layer_thickness(ds).dropna('time')
cth = cth_ds.cth

# Smooth over 5 time steps, at least 100 m thick, cloudy steps only
cth_interp = cth.rolling(time=5, center=True, min_periods=1).mean().interpolate_na('time')
thickness_interp = cth_ds.thickness.rolling(time=5, center=True, min_periods=1).mean().interpolate_na('time')
thickness_interp = thickness_interp.where(thickness_interp > 100, 100)
cloudy_steps = cth_ds.cloudmask > 0
thickness_interp = thickness_interp.sel(time=cloudy_steps)
cth_interp = cth_interp.sel(time=cloudy_steps).fillna(500)   # 500 m where no cloud top is left

cth_interp.plot(ax=ax, label='Cloud top height', color='blue')
(cth_interp - thickness_interp).plot(ax=ax, label='Cloud base height', color='orange')
ax.set_ylabel('Altitude (m)')
ax.legend()
ax2.hist(cth, bins=25)
ax2.set_xlabel('Cloud top height (m)')

## Cloudy simulation

One batch run: cloud top, base and water content are dataset variables (`cloud_top_var`, `cloud_bottom_var`, `cloud_wc_var`), so every time step gets its own layer.

In [ ]:
cloudy = track.sel(time=cth_interp.time)
cloudy['cth'] = cth_interp / 1000                         # km
cloudy['cbh'] = (cth_interp - thickness_interp) / 1000    # km
cloudy['lwc'] = xr.full_like(cloudy['cth'], 0.1)          # g m-3

print(cloudy.pyradtran.inspect_cloud_file(selector={'time': cloudy.time.values[0]},
                                          cloud_wc_var='lwc', cloud_top_var='cth', cloud_bottom_var='cbh'))

ds_sim_cloudy = cloudy.pyradtran.run(
    config_path=config_path,
    save_to_file=False,
    cloud_wc_var='lwc',
    cloud_top_var='cth',
    cloud_bottom_var='cbh',
    show_progress=False,
)
ds_sim_cloudy

## Cloud effect on the spectral irradiance

Upward and downward spectral irradiance at flight level, averaged over the cloudy time steps (mW m⁻² nm⁻¹), and the flight-level albedo $E^\uparrow / (E_\mathrm{dir} + E^\downarrow_\mathrm{diff})$. Water-vapour bands near 1.4 and 1.9 µm and the liquid-water absorption beyond 1.5 µm shape the cloudy spectra.

In [ ]:
clear = ds_sim_clear.sel(time=cloudy.time).isel(altitude=0).mean('time')
cld = ds_sim_cloudy.isel(altitude=0).mean('time')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for res, name, ls in [(clear, 'clear', '--'), (cld, 'cloudy', '-')]:
    (res.edir + res.edn).plot(ax=ax1, ls=ls, color='tab:orange', label=f'down, {name}')
    res.eup.plot(ax=ax1, ls=ls, color='tab:blue', label=f'up, {name}')
    (res.eup / (res.edir + res.edn)).plot(ax=ax2, ls=ls, color='k', label=name)
ax1.set_ylabel('Spectral irradiance (mW m⁻² nm⁻¹)')
ax2.set_ylabel('Albedo at flight level')
for a in (ax1, ax2):
    a.set_xlabel('Wavelength (nm)')
    a.set_title('')
    a.legend()
    a.grid(alpha=0.4)
plt.tight_layout()